In [ ]:
# === 1) Імпорти та фіксація seed ===
import re, time, random, numpy as np, pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# === 2) Завантаження Fake.csv / True.csv ===
FAKE_PATH = "/content/Fake.csv"
TRUE_PATH = "/content/True.csv"

fake_df = pd.read_csv(FAKE_PATH)
true_df = pd.read_csv(TRUE_PATH)

fake_df["label"] = 1
true_df["label"] = 0

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Візьмемо Title + Text (можна замінити на лише Text)
df["content"] = (df["title"].fillna("") + " " + df["text"].fillna("")).astype(str)

df[["content","label"]].head()

In [ ]:
# === 3) Мінімальне очищення тексту (lowercase + прибрати зайві символи) ===
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)           # лінки
    text = re.sub(r"[^a-z0-9\s']", " ", text)             # залишаємо букви/цифри/пробіли
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text

df["content"] = df["content"].apply(clean_text)
df["content"].head()

In [ ]:
# === 4) Токенізація + словник ===
def tokenize(text: str):
    return text.split()

# рахуємо частоти
counter = Counter()
for t in df["content"]:
    counter.update(tokenize(t))

VOCAB_SIZE = 30000  # top-k слів
most_common = counter.most_common(VOCAB_SIZE-2)

itos = ["<PAD>", "<UNK>"] + [w for w, _ in most_common]
stoi = {w:i for i,w in enumerate(itos)}

PAD_IDX = stoi["<PAD>"]
UNK_IDX = stoi["<UNK>"]

def numericalize(tokens):
    return [stoi.get(tok, UNK_IDX) for tok in tokens]

len(itos), itos[:10]

In [ ]:
# === 5) Перетворення у індекси + padding/truncation ===
MAX_LEN = 250

def encode(text: str, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = numericalize(tokens)[:max_len]
    if len(ids) < max_len:
        ids = ids + [PAD_IDX]*(max_len-len(ids))
    return ids

df["ids"] = df["content"].apply(encode)
df["ids"].iloc[0][:15], len(df["ids"].iloc[0])

In [ ]:
# === 6) Train/Val/Test split 80/10/10 ===
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

len(train_df), len(val_df), len(test_df)

In [ ]:
# === 7) Dataset/DataLoader ===
class NewsDataset(Dataset):
    def __init__(self, frame):
        self.X = frame["ids"].tolist()
        self.y = frame["label"].tolist()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.float)

BATCH_SIZE = 64

train_loader = DataLoader(NewsDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NewsDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(NewsDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)

---

## 3. Моделі

### 3.1 Simple RNN classifier
Реалізація відповідає скелету з методички: Embedding → RNN → FC → Sigmoid.

### 3.2 LSTM classifier
Embedding → LSTM (може бути bidirectional) → FC → Sigmoid.

In [ ]:
# === 8) Simple RNN Classifier ===
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity='tanh',
            dropout=(dropout if num_layers > 1 else 0.0)
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embedding(x)              # (B, L, emb_dim)
        out, h_n = self.rnn(emb)             # h_n: (layers, B, H)
        last = h_n[-1]                       # (B, H)
        logits = self.fc(last).squeeze(1)    # (B,)
        prob = torch.sigmoid(logits)
        return prob, logits

# === 9) LSTM Classifier ===
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers=1, bidirectional=False, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=(dropout if num_layers > 1 else 0.0)
        )
        factor = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * factor, 1)
        self.bidirectional = bidirectional

    def forward(self, x):
        emb = self.embedding(x)
        out, (h_n, c_n) = self.lstm(emb)
        if self.bidirectional:
            # concat forward + backward last states
            last = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            last = h_n[-1]
        logits = self.fc(last).squeeze(1)
        prob = torch.sigmoid(logits)
        return prob, logits

---

## 4. Навчання та валідація

Використовуємо:
- `BCEWithLogitsLoss()` (бо працюємо з логітами)
- `Adam`
- gradient clipping `clip_grad_norm_` для стабільності

In [ ]:
# === 10) Training / Validation loops ===
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    all_probs, all_labels = [], []
    total_loss = 0.0

    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        if train_mode:
            optimizer.zero_grad()

        probs, logits = model(x_batch)
        loss = criterion(logits, y_batch)

        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        total_loss += loss.item() * x_batch.size(0)
        all_probs.append(probs.detach().cpu())
        all_labels.append(y_batch.detach().cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    preds = (all_probs >= 0.5).astype(int)
    acc = accuracy_score(all_labels, preds)
    f1  = f1_score(all_labels, preds)
    return total_loss / len(loader.dataset), acc, f1, all_probs, all_labels

def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss":[], "val_loss":[], "train_f1":[], "val_f1":[], "train_acc":[], "val_acc":[]}

    for ep in range(1, epochs+1):
        tr_loss, tr_acc, tr_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc, va_f1, _, _ = run_epoch(model, val_loader, criterion, None)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_f1"].append(tr_f1)
        history["val_f1"].append(va_f1)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        print(f"Epoch {ep:02d}: "
              f"train loss={tr_loss:.4f}, f1={tr_f1:.4f}, acc={tr_acc:.4f} | "
              f"val loss={va_loss:.4f}, f1={va_f1:.4f}, acc={va_acc:.4f}")
    return history

In [ ]:
# === 11) Базове навчання двох моделей ===
BASE_CFG = dict(
    vocab_size=len(itos),
    emb_dim=100,
    hidden_dim=128,
    num_layers=1,
    dropout=0.2
)

rnn_model = SimpleRNNClassifier(**BASE_CFG)
lstm_model = LSTMClassifier(**BASE_CFG, bidirectional=False)

print("Training RNN...")
hist_rnn = train_model(rnn_model, train_loader, val_loader, epochs=5, lr=1e-3)

print("\nTraining LSTM...")
hist_lstm = train_model(lstm_model, train_loader, val_loader, epochs=5, lr=1e-3)

In [ ]:
# === 12) Графіки train/val loss і F1 (вимога методички) ===
def plot_history(hist, title):
    epochs = range(1, len(hist["train_loss"])+1)

    plt.figure()
    plt.plot(epochs, hist["train_loss"], label="train")
    plt.plot(epochs, hist["val_loss"], label="val")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
    plt.show()

    plt.figure()
    plt.plot(epochs, hist["train_f1"], label="train")
    plt.plot(epochs, hist["val_f1"], label="val")
    plt.title(f"{title} - F1")
    plt.xlabel("Epoch"); plt.ylabel("F1"); plt.legend(); plt.grid(True)
    plt.show()

plot_history(hist_rnn, "Simple RNN")
plot_history(hist_lstm, "LSTM")

In [ ]:
# === 13) Оцінка на тесті + Confusion Matrix + ROC ===
def evaluate_on_test(model, loader):
    criterion = nn.BCEWithLogitsLoss()
    loss, acc, f1, probs, labels = run_epoch(model, loader, criterion, None)
    preds = (probs >= 0.5).astype(int)

    prec = precision_score(labels, preds)
    rec  = recall_score(labels, preds)
    auc  = roc_auc_score(labels, probs)

    return {
        "loss": loss, "acc": acc, "prec": prec, "rec": rec, "f1": f1, "auc": auc,
        "probs": probs, "labels": labels, "preds": preds
    }

res_rnn  = evaluate_on_test(rnn_model, test_loader)
res_lstm = evaluate_on_test(lstm_model, test_loader)

res_rnn, res_lstm

In [ ]:
def plot_confusion_and_roc(res, title):
    labels, preds, probs = res["labels"], res["preds"], res["probs"]

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{title} - Confusion Matrix")
    plt.colorbar()
    plt.xticks([0,1], ["Real(0)", "Fake(1)"])
    plt.yticks([0,1], ["Real(0)", "Fake(1)"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.show()

    # ROC curve
    fpr, tpr, _ = roc_curve(labels, probs)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {res['auc']:.4f}")
    plt.plot([0,1], [0,1], linestyle="--")
    plt.title(f"{title} - ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(); plt.grid(True)
    plt.show()

plot_confusion_and_roc(res_rnn, "Simple RNN")
plot_confusion_and_roc(res_lstm, "LSTM")

---

## 5. Пошук гіперпараметрів (Part E)

Для демонстрації робимо невеликий grid-search.
Основна метрика порівняння — **F1-score**.
Також виводимо ROC AUC, час на епоху і кількість параметрів.

In [ ]:
# === 14) Hyperparameter Search (скорочений) ===
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def train_config(model_cls, cfg, epochs=3):
    set_seed(42)
    model = model_cls(**cfg).to(device)
    start = time.time()
    hist = train_model(model, train_loader, val_loader, epochs=epochs, lr=cfg.get("lr", 1e-3))
    epoch_time = (time.time() - start) / epochs

    res = evaluate_on_test(model, test_loader)
    return {
        "model": model_cls.__name__,
        "emb_dim": cfg["emb_dim"],
        "hidden_dim": cfg["hidden_dim"],
        "layers": cfg["num_layers"],
        "bidirectional": cfg.get("bidirectional", False),
        "dropout": cfg["dropout"],
        "lr": cfg.get("lr", 1e-3),
        "params": count_params(model),
        "f1": res["f1"],
        "auc": res["auc"],
        "epoch_time": epoch_time
    }

grid = [
    dict(vocab_size=len(itos), emb_dim=50,  hidden_dim=64,  num_layers=1, dropout=0.2, lr=1e-3),
    dict(vocab_size=len(itos), emb_dim=100, hidden_dim=128, num_layers=1, dropout=0.2, lr=1e-3),
    dict(vocab_size=len(itos), emb_dim=200, hidden_dim=256, num_layers=2, dropout=0.5, lr=5e-4),
]

results = []
for cfg in grid:
    results.append(train_config(SimpleRNNClassifier, cfg, epochs=3))
    results.append(train_config(LSTMClassifier, {**cfg, "bidirectional": False}, epochs=3))
    results.append(train_config(LSTMClassifier, {**cfg, "bidirectional": True}, epochs=3))

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
results_df